In [1]:
!pip install torchmetrics --quiet

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch

print(torch.__version__)

import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data_utils

from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy

from sklearn.preprocessing import LabelEncoder

2.11.0+cpu


In [3]:
INPUT_SIZE = 36
HIDDEN_SIZE = 25
OUTPUT_SIZE = 5
LEARNING_RATE = 1e-2
EPOCHS = 20
BATCH_SIZE = 256
EMBEDDING_SIZE = 5

In [4]:
class CustomDataset(Dataset):
    def __init__(self):
        X = pd.read_csv("X_cat.csv", sep="\t", index_col=0)
        target = pd.read_csv("y_cat.csv", sep="\t", index_col=0, names=["status"])

        weekday_columns = [f"Weekday_{i}" for i in range(7)]
        weekdays = np.argmax(X[weekday_columns].values, axis=1)

        X.drop(weekday_columns, axis=1, inplace=True)


        X["Weekday_cos"] = np.cos((2 * np.pi / 7.) * weekdays)
        X["Weekday_sin"] = np.sin((2 * np.pi / 7.) * weekdays)

        X["Hour_cos"] = np.cos((2 * np.pi / 24.) * X["Hour"].values)
        X["Hour_sin"] = np.sin((2 * np.pi / 24.) * X["Hour"].values)

        X["Month_cos"] = np.cos((2 * np.pi / 12.) * X["Month"].values)
        X["Month_sin"] = np.sin((2 * np.pi / 12.) * X["Month"].values)

        X["Gender"] = np.argmax(X[["Sex_Female", "Sex_Male", "Sex_Unknown"]].values, axis=1)

        X.drop(["Sex_Female", "Sex_Male", "Sex_Unknown"], axis=1, inplace=True)

        target = target.iloc[:, :].values
        target[target == "Died"] = "Euthanasia"

        le = LabelEncoder()
        self.y = le.fit_transform(target)

        self.X = X.values

        self.columns = X.columns.values

        self.embedding_column = "Gender"
        self.nrof_emb_categories = 3
        self.numeric_columns = [
            "IsDog",
            "Age",
            "HasName",
            "NameLength",
            "NameFreq",
            "MixColor",
            "ColorFreqAsIs",
            "ColorFreqBase",
            "TabbyColor",
            "MixBreed",
            "Domestic",
            "Shorthair",
            "Longhair",
            "Year",
            "Day",
            "Breed_Chihuahua Shorthair Mix",
            "Breed_Domestic Medium Hair Mix",
            "Breed_Domestic Shorthair Mix",
            "Breed_German Shepherd Mix",
            "Breed_Labrador Retriever Mix",
            "Breed_Pit Bull Mix",
            "Breed_Rare",
            "SexStatus_Flawed",
            "SexStatus_Intact",
            "SexStatus_Unknown",
            "Weekday_cos",
            "Weekday_sin",
            "Hour_cos",
            "Hour_sin",
            "Month_cos",
            "Month_sin",
        ]

    def __len__(self):
        return len(self.X)

    # Переопределяем метод, который достает по индексу наблюдение из датасет
    def __getitem__(self, idx):
        row = self.X[idx, :]
        row = {col: torch.tensor(row[i]) for i, col in enumerate(self.columns)}
        return row, self.y[idx]

In [5]:
class MLPNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, nrof_cat, emb_dim, emb_columns, numeric_columns):
        super(MLPNet, self).__init__()
        self.emb_columns = emb_columns
        self.numeric_columns = numeric_columns

        self.emb_layer = torch.nn.Embedding(nrof_cat, emb_dim)

        self.feature_bn = torch.nn.BatchNorm1d(input_size)

        self.linear1 = torch.nn.Linear(input_size, hidden_size)
        self.linear1.apply(self.init_weights)
        self.bn1 = torch.nn.BatchNorm1d(hidden_size)

        self.linear2 = torch.nn.Linear(hidden_size, hidden_size)
        self.linear2.apply(self.init_weights)
        self.bn2 = torch.nn.BatchNorm1d(hidden_size)

        self.linear3 = torch.nn.Linear(hidden_size, output_size)

    def init_weights(self, m):
        if type(m) == nn.Linear:
            torch.nn.init.xavier_uniform(m.weight)
            # m.bias.data.fill_(0.001)

    def forward(self, x):
        emb_output = self.emb_layer(torch.tensor(x[self.emb_columns], dtype=torch.int64))
        numeric_feats = torch.tensor(pd.DataFrame(x)[self.numeric_columns].values, dtype=torch.float32)

        concat_input = torch.cat([numeric_feats, emb_output], dim=1)
        output = self.feature_bn(concat_input)

        output = self.linear1(output)
        output = self.bn1(output)
        output = torch.relu(output)

        output = self.linear2(output)
        output = self.bn2(output)
        output = torch.relu(output)

        output = self.linear3(output)

        return output

In [6]:
def run_train(model, train_loader):
    step = 0
    for epoch in range(EPOCHS):
        model.train()

        for features, label in train_loader:
            # Reset gradients
            optimizer.zero_grad()

            output = model(features)
            # Calculate error and backpropagate
            loss = criterion(output, label)
            loss.backward()
            acc = accuracy(output, label).item()

            # Update weights with gradients
            optimizer.step()

            step += 1

            if step % 100 == 0:
                print(f"EPOCH {epoch} STEP {step} : train_loss: {loss.item()} train_acc: {acc}")

    return step

In [7]:
animal_dataset = CustomDataset()
train_loader = data_utils.DataLoader(
    dataset=animal_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

model = MLPNet(
    INPUT_SIZE,
    HIDDEN_SIZE,
    OUTPUT_SIZE,
    animal_dataset.nrof_emb_categories,
    EMBEDDING_SIZE,
    animal_dataset.embedding_column,
    animal_dataset.numeric_columns,
)

criterion = nn.CrossEntropyLoss()
accuracy = Accuracy(task="multiclass", num_classes=OUTPUT_SIZE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [8]:
step = run_train(model, train_loader)

EPOCH 0 STEP 100 : train_loss: 0.773709237575531 train_acc: 0.69921875
EPOCH 1 STEP 200 : train_loss: 0.8603692054748535 train_acc: 0.6171875
EPOCH 2 STEP 300 : train_loss: 0.7927767634391785 train_acc: 0.65234375
EPOCH 3 STEP 400 : train_loss: 0.7692281007766724 train_acc: 0.703125
EPOCH 4 STEP 500 : train_loss: 0.7582111954689026 train_acc: 0.703125
EPOCH 5 STEP 600 : train_loss: 0.8095897436141968 train_acc: 0.63671875
EPOCH 6 STEP 700 : train_loss: 0.7136396169662476 train_acc: 0.69140625
EPOCH 7 STEP 800 : train_loss: 0.7304157614707947 train_acc: 0.6953125
EPOCH 8 STEP 900 : train_loss: 0.7433536648750305 train_acc: 0.69140625
EPOCH 9 STEP 1000 : train_loss: 0.776448667049408 train_acc: 0.65625
EPOCH 10 STEP 1100 : train_loss: 0.7483952641487122 train_acc: 0.703125
EPOCH 11 STEP 1200 : train_loss: 0.8293687701225281 train_acc: 0.66015625
EPOCH 12 STEP 1300 : train_loss: 0.7459456324577332 train_acc: 0.67578125
EPOCH 13 STEP 1400 : train_loss: 0.8051178455352783 train_acc: 0.64062